In [ ]:
# Mini IDS Visualization Notebook

This notebook accompanies the Mini IDS project. It demonstrates how to build a simple IDS using Scapy, capture packets, detect attacks, log events, and verify using Wireshark. It also includes tools to visualize the CSV log produced by the IDS.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Import Required Libraries

Import necessary Python libraries such as Scapy, datetime, csv, and os for packet manipulation, logging, and file operations.
</VSCode.Cell>
<VSCode.Cell language="python">
from scapy.all import sniff, IP, TCP, UDP, ICMP
import os, csv, time
from datetime import datetime
from collections import defaultdict, deque
import pandas as pd
import matplotlib.pyplot as plt

# ensure inline plotting (if running in VS Code Jupyter)
%matplotlib inline
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Network Configuration Steps

Provide steps to configure VMWare Workstation for Bridged or NAT networking to allow packet capture between Windows host and Kali Linux VM.

- In VMware Workstation, open VM settings > Network Adapter.
- Choose **Bridged** to make the VM appear on the same LAN as the host (useful for scanning the local network).  
- Choose **NAT** for outbound-only connectivity; the VM is behind a virtual NAT.
- After changing mode, restart the VM and verify network with `ip a` on Kali.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Required Installations

List and explain installation commands for Python, Scapy, Npcap (on Windows), and Wireshark on both Windows and Kali Linux.

- Kali (VM):
```bash
sudo apt update && sudo apt install -y python3 python3-pip
sudo pip3 install scapy pandas matplotlib
```
- Windows host (optional):
  - Install Python 3 and pip.
  - Install Npcap from https://nmap.org/npcap/ (needed by Scapy on Windows).
  - Install Wireshark for packet capture/verification.

You may also install `jupyter` or use VS Code's notebook support.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Folder Structure Setup

Define and create a folder structure for the project, including directories for logs, scripts, and configuration files.
</VSCode.Cell>
<VSCode.Cell language="python">
import os

base = os.getcwd()
print("Base directory", base)
for d in ["logs","mini_ids"]:
    os.makedirs(d, exist_ok=True)

print("Created/verified structure")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Packet Capture Function

Define a function using Scapy to capture live network packets, filtering for TCP, UDP, and ICMP protocols.
</VSCode.Cell>
<VSCode.Cell language="python">
def packet_record(pkt):
    rec = {"timestamp":datetime.utcnow().isoformat(),
           "src":"","dst":"","proto":"","sport":"","dport":"","len":len(pkt)}
    if IP in pkt:
        rec["src"] = pkt[IP].src
        rec["dst"] = pkt[IP].dst
        if TCP in pkt:
            rec.update({"proto":"TCP","sport":pkt[TCP].sport,"dport":pkt[TCP].dport})
        elif UDP in pkt:
            rec.update({"proto":"UDP","sport":pkt[UDP].sport,"dport":pkt[UDP].dport})
        elif ICMP in pkt:
            rec.update({"proto":"ICMP"})
    return rec

print(packet_record(IP(src="1.1.1.1",dst="2.2.2.2")/TCP()))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. Detection Functions for Port Scanning

Implement a modular function to detect port scanning by monitoring sequential port access attempts from a single IP.
</VSCode.Cell>
<VSCode.Cell language="python">
PORT_THRESHOLD=20

ports_seen=defaultdict(set)
def detect_port_scan(src,dport):
    ports_seen[src].add(dport)
    if len(ports_seen[src])>=PORT_THRESHOLD:
        return True
    return False

# quick demo
print(detect_port_scan("1.1.1.1",80))
for p in range(1,21):
    detect_port_scan("1.1.1.1",p)
print("scan?", detect_port_scan("1.1.1.1",22))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 7. Detection Functions for SYN Flood Attacks

Create a function to identify SYN flood attacks by tracking incomplete TCP handshakes and exceeding a SYN packet threshold.
</VSCode.Cell>
<VSCode.Cell language="python">
syn_times=defaultdict(lambda:deque())
SYN_THRESHOLD=200

import time

def detect_syn(src):
    now=time.time()
    syn_times[src].append(now)
    # prune older than 10s
    while syn_times[src] and syn_times[src][0]<now-10:
        syn_times[src].popleft()
    return len(syn_times[src])>=SYN_THRESHOLD

# fake test
for i in range(210):
    if detect_syn("1.1.1.1"):
        print("syn flood detected at",i)
        break
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 8. Detection Functions for ICMP Flood Attacks

Develop a function to detect ICMP (ping) floods by counting ICMP echo requests above a defined rate threshold.
</VSCode.Cell>
<VSCode.Cell language="python">
icmp_times=defaultdict(lambda:deque())
ICMP_THRESHOLD=200

def detect_icmp(src):
    now=time.time()
    icmp_times[src].append(now)
    while icmp_times[src] and icmp_times[src][0]<now-10:
        icmp_times[src].popleft()
    return len(icmp_times[src])>=ICMP_THRESHOLD

# quick demo
for i in range(205):
    if detect_icmp("1.1.1.1"):
        print("icmp flood at",i)
        break
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 9. Detection Functions for Suspicious IP Activity

Build a function to flag suspicious IP activity based on multiple requests exceeding a threshold from the same source IP.
</VSCode.Cell>
<VSCode.Cell language="python">
total_times=defaultdict(lambda:deque())
SUSPICIOUS_THRESHOLD=300

def detect_suspicious(src):
    now=time.time()
    total_times[src].append(now)
    while total_times[src] and total_times[src][0]<now-10:
        total_times[src].popleft()
    return len(total_times[src])>=SUSPICIOUS_THRESHOLD

# test
for i in range(305):
    if detect_suspicious("1.1.1.1"):
        print("suspicious at",i)
        break
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 10. Anomaly Detection Using Packet Rate Monitoring

Add basic anomaly detection by calculating packet rates over time intervals and alerting on deviations from normal baselines.
</VSCode.Cell>
<VSCode.Cell language="python">
global_times=deque()
RATE_THRESHOLD=1000

def global_rate():
    now=time.time()
    global_times.append(now)
    while global_times and global_times[0]<now-10:
        global_times.popleft()
    return len(global_times)

# simulate
for i in range(1100):
    r=global_rate()
    if r>=RATE_THRESHOLD:
        print('high rate',r,'at packet',i)
        break
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 11. Logging to TXT and CSV Files

Implement logging functions to write detection events to a .txt file and generate a simple CSV log for visualization purposes.
</VSCode.Cell>
<VSCode.Cell language="python">
LOG_DIR='../logs'
TEXT_LOG=os.path.join(LOG_DIR,'alerts.txt')
CSV_LOG=os.path.join(LOG_DIR,'packets.csv')

os.makedirs(LOG_DIR,exist_ok=True)

import csv

def log_alert(text):
    ts=datetime.utcnow().isoformat()
    with open(TEXT_LOG,'a') as f:
        f.write(f"[{ts}] {text}\n")
    print(text)

def log_packet(rec):
    exists=os.path.exists(CSV_LOG)
    with open(CSV_LOG,'a',newline='') as f:
        writer=csv.writer(f)
        if not exists:
            writer.writerow(['timestamp','src','dst','proto','sport','dport','len'])
        writer.writerow([rec.get(k,'') for k in ['timestamp','src','dst','proto','sport','dport','len']])

# demo record
log_packet(packet_record(IP(src='1.1.1.1',dst='2.2.2.2')/TCP()))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 12. Real-Time Alerts in Terminal

Define a function to display real-time alerts in the terminal with timestamps and detection details.
</VSCode.Cell>
<VSCode.Cell language="python">
def alert_now(msg):
    print(f"[{datetime.utcnow().isoformat()}] ALERT: {msg}")

alert_now('test alert')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 13. Main IDS Execution Loop

Create the main loop to run the IDS, capturing packets, applying detection logic, logging, and alerting, with instructions to run as admin.
</VSCode.Cell>
<VSCode.Cell language="python">
def handle_pkt(pkt):
    rec=packet_record(pkt)
    log_packet(rec)
    if IP in pkt:
        src=pkt[IP].src
        if TCP in pkt and pkt[TCP].flags & 0x02 and not pkt[TCP].flags & 0x10:
            if detect_syn(src): alert_now(f"SYN flood from {src}")
        if ICMP in pkt:
            if detect_icmp(src): alert_now(f"ICMP flood from {src}")
        if TCP in pkt or UDP in pkt:
            if detect_port_scan(src,pkt.sport if TCP in pkt else pkt[UDP].dport):
                alert_now(f"Port scan from {src}")
        if detect_suspicious(src): alert_now(f"Suspicious activity {src}")
        if global_rate()>=RATE_THRESHOLD:
            alert_now("High global rate")

print('NOTE: run this cell only as root/admin within an environment where sniffing is permitted')

# sniff(iface=None, prn=handle_pkt, store=False)  # uncomment when running live
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 14. Sample Attack Simulation Commands

Provide sample commands to simulate attacks from Kali Linux, such as using nmap for port scanning, hping3 for SYN floods, and ping for ICMP floods.
```bash
nmap -sS -p1-2000 TARGET_IP
sudo hping3 -S --flood -V -p 80 TARGET_IP
sudo ping -f -c 10000 TARGET_IP
```
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 15. Testing Methodology

Outline steps to test the IDS, including running simulations, verifying logs, and checking alerts in a controlled environment.

1. Run the IDS as root in the VM.
2. Start Wireshark capture for verification.
3. Launch attack commands from another shell or machine.
4. Observe alerts, inspect `logs/alerts.txt` and `logs/packets.csv`.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 16. Wireshark Verification Steps

Explain how to use Wireshark to capture and verify packets during testing, comparing with IDS detections.
- Choose the same interface used by the IDS.
- Apply filters such as `tcp.flags.syn==1 && tcp.flags.ack==0` or `icmp`.
- Match captured packets against entries in `logs/packets.csv`.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 17. Future Improvements

Suggest enhancements like integrating machine learning for advanced anomaly detection or adding support for more protocols.

- Add async logging queue to avoid drops under high load.
- Geo-IP / DNS enrichment.
- Web dashboard with real-time charts.
- Machine learning model for anomaly detection on packet features.
